# 🛠️ Chapter 04-01b: Hands-On — Build a Semantic Similarity Engine

---

## From Words to Vectors to Search! 🔍

In this notebook, you'll generate real embeddings and build a working similarity engine.

By the end:
- ✅ Generated embeddings with OpenAI
- ✅ Computed cosine similarity between sentences
- ✅ Built a semantic search engine
- ✅ Visualized embeddings with PCA
- ✅ Explored embedding arithmetic (King - Man + Woman)

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install openai python-dotenv numpy scikit-learn matplotlib

In [ ]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Step 1: Generate Your First Embedding

In [ ]:
# Generate a single embedding
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Python is a great programming language"
)

embedding = response.data[0].embedding

print(f"📐 Dimensions: {len(embedding)}")
print(f"📊 First 10 values: {embedding[:10]}")
print(f"🔢 Type: list of {type(embedding[0]).__name__}")

In [ ]:
# Helper function: get embedding for any text
def get_embedding(text: str, model: str = "text-embedding-3-small") -> list:
    """Get embedding vector for a text string."""
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

# Batch version — more efficient!
def get_embeddings(texts: list, model: str = "text-embedding-3-small") -> list:
    """Get embeddings for multiple texts in one API call."""
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

print("✅ Helper functions ready!")

---

## 🧪 Step 2: Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity(text1: str, text2: str) -> float:
    """Compute cosine similarity between two texts."""
    emb1 = get_embedding(text1)
    emb2 = get_embedding(text2)
    return cosine_similarity([emb1], [emb2])[0][0]

# Test with similar and different texts
pairs = [
    ("I love programming in Python", "Python is my favorite coding language"),
    ("I love programming in Python", "The weather is sunny today"),
    ("Machine learning is fascinating", "AI and deep learning are interesting"),
    ("Machine learning is fascinating", "I had pasta for dinner"),
]

print("📏 Cosine Similarity Scores:")
print("=" * 70)
for t1, t2 in pairs:
    score = compute_similarity(t1, t2)
    bar = "█" * int(score * 30)
    print(f"\n  \"{t1[:40]}...\"")
    print(f"  \"{t2[:40]}...\"")
    print(f"  Score: {score:.4f} {bar}")

---

## 🚀 Step 3: Semantic Search Engine

In [ ]:
# Knowledge base — imagine these are your documents
documents = [
    "Python is a high-level programming language known for its readability.",
    "Machine learning models learn patterns from data to make predictions.",
    "React is a JavaScript library for building user interfaces.",
    "Neural networks are inspired by the human brain's structure.",
    "Docker containers package applications with their dependencies.",
    "SQL is used to query and manage relational databases.",
    "Transformers use attention mechanisms for natural language processing.",
    "Kubernetes orchestrates container deployments at scale.",
    "RAG combines retrieval with generation for better AI answers.",
    "Git is a version control system for tracking code changes.",
    "LangChain is a framework for building LLM-powered applications.",
    "Vector databases store embeddings for fast similarity search.",
]

# Pre-compute all document embeddings (one API call!)
doc_embeddings = get_embeddings(documents)
print(f"✅ Embedded {len(documents)} documents ({len(doc_embeddings[0])} dimensions each)")

In [ ]:
def semantic_search(query: str, top_k: int = 3) -> list:
    """Search documents by semantic meaning."""
    query_emb = get_embedding(query)
    
    # Compute similarity with all documents
    similarities = cosine_similarity([query_emb], doc_embeddings)[0]
    
    # Get top-k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "score": similarities[idx]
        })
    return results

# Test semantic search
queries = [
    "How do I build AI applications?",
    "What's a good tool for managing containers?",
    "Tell me about brain-inspired computing",
]

for q in queries:
    print(f"\n🔍 Query: \"{q}\"")
    results = semantic_search(q)
    for i, r in enumerate(results, 1):
        print(f"   {i}. [{r['score']:.4f}] {r['document']}")
    print("-" * 70)

---

## 📊 Step 4: Visualize Embeddings with PCA

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Reduce 1536 dimensions → 2 for visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(doc_embeddings)

# Color categories
categories = [
    "programming", "AI/ML", "web", "AI/ML",
    "devops", "data", "AI/ML", "devops",
    "AI/ML", "devops", "AI/ML", "AI/ML"
]
color_map = {"programming": "blue", "AI/ML": "red", "web": "green", "devops": "orange", "data": "purple"}
colors = [color_map[c] for c in categories]

plt.figure(figsize=(12, 8))
for i, (x, y) in enumerate(coords):
    plt.scatter(x, y, c=colors[i], s=100, zorder=5)
    label = documents[i][:35] + "..."
    plt.annotate(label, (x, y), fontsize=7, ha='left', va='bottom')

# Legend
for cat, color in color_map.items():
    plt.scatter([], [], c=color, label=cat, s=100)
plt.legend(title="Category")
plt.title("Document Embeddings Visualized (PCA 2D)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Notice: AI/ML docs cluster together, DevOps docs cluster together!")

---

## 🧪 Step 5: Embedding Arithmetic

In [ ]:
# King - Man + Woman ≈ Queen?

words = ["king", "man", "woman", "queen", "prince", "princess", "boy", "girl"]
word_embeddings = {w: np.array(get_embedding(w)) for w in words}

# Arithmetic: king - man + woman = ???
result_vector = word_embeddings["king"] - word_embeddings["man"] + word_embeddings["woman"]

# Find the closest word
print("👑 King - Man + Woman = ???\n")
for word, emb in word_embeddings.items():
    sim = cosine_similarity([result_vector], [emb])[0][0]
    bar = "█" * int(sim * 30)
    marker = " 👈 CLOSEST!" if word == "queen" else ""
    print(f"  {word:12s} → {sim:.4f} {bar}{marker}")

---

## 🧪 Step 6: Similarity Heatmap

In [ ]:
# Build a heatmap of document similarities

sim_matrix = cosine_similarity(doc_embeddings)

plt.figure(figsize=(10, 8))
plt.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(label='Cosine Similarity')

labels = [d[:30] + "..." for d in documents]
plt.xticks(range(len(labels)), labels, rotation=45, ha='right', fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.title("Document Similarity Heatmap")
plt.tight_layout()
plt.show()

print("💡 Green = similar. Red = different. Diagonal = 1.0 (self-similarity).")

---

## ✍️ Practice: FAQ Matcher

**Challenge:** Build a FAQ system that matches user questions to pre-defined answers.

In [ ]:
# YOUR CODE HERE

faq_data = {
    "How do I reset my password?": "Go to Settings → Security → Reset Password.",
    "What are your business hours?": "We're open Mon-Fri, 9 AM to 6 PM EST.",
    "How do I cancel my subscription?": "Go to Billing → Cancel Subscription.",
    "Do you offer refunds?": "Yes, within 30 days of purchase.",
    "How do I contact support?": "Email support@example.com or use live chat.",
}

# TODO: Embed all FAQ questions
# TODO: Write a function match_faq(user_question) that finds the best match
# TODO: Test with:
#   "I forgot my login credentials"
#   "When are you guys open?"
#   "I want my money back"

<details>
<summary>💡 Click to see solution</summary>

```python
faq_questions = list(faq_data.keys())
faq_answers = list(faq_data.values())
faq_embeddings = get_embeddings(faq_questions)

def match_faq(user_question: str) -> tuple:
    query_emb = get_embedding(user_question)
    sims = cosine_similarity([query_emb], faq_embeddings)[0]
    best_idx = np.argmax(sims)
    return faq_questions[best_idx], faq_answers[best_idx], sims[best_idx]

for q in ["I forgot my login credentials", "When are you guys open?", "I want my money back"]:
    matched_q, answer, score = match_faq(q)
    print(f"Q: {q}")
    print(f"Matched: {matched_q} (score: {score:.4f})")
    print(f"A: {answer}\n")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 get_embedding() → 1536 numbers capturing meaning.             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Batch with get_embeddings() for efficiency.                    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 cosine_similarity = go-to metric for text search.             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Pre-compute + store embeddings for fast search at scale.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 PCA for visualization; real search stays high-dimensional.     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Vector Databases!

Our search engine works but lives in memory. Vector DBs (ChromaDB, FAISS, Pinecone) persist and scale to millions.

---

### 🎉 You Built a Search Engine!
No keyword matching — pure meaning-based search. This is the heart of modern AI search. 🚀